<!--nav--> [🗺 Learning path](README.md) · **16/46** · ◀ [Multi Trace Agent Evaluation](./Multi_Trace_Agent_Evaluation.ipynb) · [Multimodal LoRA QLoRA DPO](./Multimodal_LoRA_QLoRA_DPO.ipynb) ▶

# Simple Multi-GPU Multimodal Training

Fine-tune a vision-language model on image+text data using multiple GPUs.

- **Model:** Qwen2-VL-2B-Instruct (2B params — small enough for free GPUs)
- **Method:** LoRA + DeepSpeed ZeRO-2 + Accelerate
- **Data:** Small set of image-description pairs
- **Platform:** Kaggle 2x T4 (free) or Colab T4 (free)

In [ ]:
!pip install -q transformers datasets peft accelerate deepspeed qwen-vl-utils

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

## Write Configs

In [ ]:
# Accelerate config — accelerate-managed DeepSpeed (no config file conflict)
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Config written. {NUM_GPUS} GPU(s), ZeRO-2, bf16.")

## Write Training Script

Fine-tune Qwen2-VL-2B on image captioning. The model sees images and learns to describe them.

In [ ]:
%%writefile train_multimodal.py
"""Distributed multimodal fine-tuning: Qwen2-VL-2B + LoRA + DeepSpeed."""
import torch, os
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen2-VL-2B-Instruct"

# Load model + processor
processor = AutoProcessor.from_pretrained(MODEL)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, dtype=torch.bfloat16,
)

# Add LoRA to language model layers only
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

# Load a small image-text dataset
dataset = load_dataset("laion/220k-GPT4Vision-captions-from-LIVIS", split="train")
dataset = dataset.shuffle(seed=42).select(range(200))
print(f"Dataset: {len(dataset)} image-text pairs")


def preprocess(examples):
    """Format each example as a chat with an image, then tokenize."""
    texts = []
    images = []
    for img, caption in zip(examples["image"], examples["caption"]):
        if img is None or img.mode != "RGB":
            # Skip bad images — will be filtered out
            texts.append(None)
            images.append(None)
            continue
        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": "Describe this image in detail."},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": caption},
            ]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
        images.append(img)

    # Filter out None entries
    valid = [(t, i) for t, i in zip(texts, images) if t is not None]
    if not valid:
        return {"input_ids": [], "attention_mask": [], "labels": [], "pixel_values": [], "image_grid_thw": []}
    valid_texts, valid_images = zip(*valid)

    batch = processor(
        text=list(valid_texts),
        images=list(valid_images),
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    batch["labels"] = batch["input_ids"].clone()
    # Mask padding tokens in labels
    batch["labels"][batch["labels"] == processor.tokenizer.pad_token_id] = -100

    # Convert to lists for dataset storage
    return {
        "input_ids": batch["input_ids"].tolist(),
        "attention_mask": batch["attention_mask"].tolist(),
        "labels": batch["labels"].tolist(),
        "pixel_values": batch["pixel_values"].tolist(),
        "image_grid_thw": batch["image_grid_thw"].tolist(),
    }


dataset = dataset.map(
    preprocess,
    batched=True,
    batch_size=4,
    remove_columns=dataset.column_names,
)
dataset.set_format("torch")
print(f"Processed dataset: {len(dataset)} examples")

# Train
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./vl_output",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
        dataloader_pin_memory=False,
    ),
    train_dataset=dataset,
)

trainer.train()
trainer.save_model("./vl_output/final")
processor.save_pretrained("./vl_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\n" + "=" * 50)
    print("Multimodal training complete!")
    print("=" * 50)

## Launch Training

In [ ]:
print(f"Launching multimodal training on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train_multimodal.py

elapsed = time.time() - start
print(f"\nDone! {elapsed:.0f}s on {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

## Test: Describe an Image

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from datasets import load_dataset
import torch

# Load fine-tuned model
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", dtype=torch.bfloat16
).to("cuda")
# Load LoRA weights on top
from peft import PeftModel
model = PeftModel.from_pretrained(model, "./vl_output/final")
model.eval()

processor = AutoProcessor.from_pretrained("./vl_output/final")

# Grab a test image from the dataset
test_data = load_dataset("laion/220k-GPT4Vision-captions-from-LIVIS", split="train")
test_img = test_data.shuffle(seed=99).select(range(1))[0]["image"]
display(test_img.resize((300, 300)))

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Describe this image in detail."},
    ]},
]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[test_img], return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

response = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Model: {response.strip()}")

---

## How It Works

```
accelerate launch --num_processes=N train_multimodal.py
        |
   Worker 0 (GPU 0)       Worker 1 (GPU 1)
        |                       |
   Qwen2-VL-2B             Qwen2-VL-2B
   [Vision Encoder]        [Vision Encoder]
   [Language Model]        [Language Model]
        |                       |
   LoRA on q_proj, v_proj  LoRA on q_proj, v_proj
        |                       |
   DeepSpeed ZeRO-2: optimizer sharded + CPU offload
```

### Key Differences from Text-Only Training

| Aspect | Text-Only | Multimodal |
|--------|-----------|------------|
| **Input** | Token IDs | Token IDs + pixel values + image grid |
| **Processor** | Tokenizer | AutoProcessor (tokenizer + image processor) |
| **Batch size** | 4 | 1 (images use more memory) |
| **Model** | GPT-2 (124M) | Qwen2-VL-2B (2B, has vision encoder) |

Everything else is the same: LoRA, DeepSpeed, Accelerate launcher.

| Platform | GPUs | Cost |
|----------|------|------|
| **Kaggle** | 2x T4 | Free (30h/week) |
| **Colab** | 1x T4 | Free |